# 第 8 章 基于 LeRobot 的低成本机械臂 SO-101 的应用


## 8.2.1 LeRobot 的安装


LeRobot 需要 Python 3.10 及以上版本。下面以使用 Miniconda 创建虚拟环境为例介绍安装步骤。

（1）下载并安装 Miniconda 后，创建名为 lerobot 的虚拟环境并激活：



In [ ]:
conda create -y -n lerobot python=3.10
conda activate lerobot


（2）当使用 Miniconda 时，需要在环境中安装 ffmpeg，用于视频编码与解码：



In [ ]:
conda install ffmpeg -c conda-forge


（3）克隆 LeRobot 源代码并安装。安装 SO-101 所需的 Feetech 舵机支持，使用 feetech 附加依赖：



In [ ]:
git clone https://github.com/huggingface/lerobot.git
cd lerobot
pip install -e ".[feetech]"


如果编译过程中出现错误，可能需要先安装 cmake、build-essential 以及 ffmpeg 相关库。在 Linux 系统上可执行：



In [ ]:
sudo apt-get install cmake build-essential python3-dev pkg-config \
  libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev \
  libswscale-dev libswresample-dev libavfilter-dev


## 8.2.2 USB 端口检测与舵机配置


SO-101 的两只手臂各连接一个 USB 串行舵机控制器。在开始配置之前，需要确定每只手臂对应的 USB 端口。LeRobot 提供了检测脚本：



In [ ]:
python lerobot/scripts/find_motors_bus_port.py


运行该脚本时，脚本会先列出当前检测到的所有可用端口，然后提示拔掉其中一只手臂的 USB 线并回车，从而确定该端口属于哪只手臂。在 macOS 上，端口名称通常形如 `/dev/tty.usbmodem575E0031751`；在 Linux 上通常为 `/dev/ttyACM0`、`/dev/ttyACM1` 等。Linux 系统下若出现权限问题，可先执行：



In [ ]:
sudo chmod 666 /dev/ttyACM0
sudo chmod 666 /dev/ttyACM1


得到两只手臂的端口后，需要修改 LeRobot 配置文件中 `SO101RobotConfig` 的 `port` 默认值，将主臂（leader）与从臂（follower）的端口分别填写到对应的 `MotorsBusConfig` 中：



In [ ]:
leader_arms: dict[str, MotorsBusConfig] = field(
    default_factory=lambda: {
        "main": FeetechMotorsBusConfig(
            port="{ADD YOUR LEADER PORT}",   # 修改为主臂端口
            motors={
                # name: (index, model)
                "shoulder_pan": [1, "sts3215"],
                "shoulder_lift": [2, "sts3215"],
                "elbow_flex": [3, "sts3215"],
                "wrist_flex": [4, "sts3215"],
                "wrist_roll": [5, "sts3215"],
                "gripper": [6, "sts3215"],
            },
        ),
    }
)

follower_arms: dict[str, MotorsBusConfig] = field(
    default_factory=lambda: {
        "main": FeetechMotorsBusConfig(
            port="{ADD YOUR FOLLOWER PORT}", # 修改为从臂端口
            motors={
                # name: (index, model)
                "shoulder_pan": [1, "sts3215"],
                "shoulder_lift": [2, "sts3215"],
                "elbow_flex": [3, "sts3215"],
                "wrist_flex": [4, "sts3215"],
                "wrist_roll": [5, "sts3215"],
                "gripper": [6, "sts3215"],
            },
        ),
    }
)


python lerobot/scripts/configure_motor.py \    --port /dev/tty.usbmodem58760432961 \    --brand feetech \    --model sts3215 \    --baudrate 1000000 \    --ID 1


## 8.2.3 舵机标定


标定（Calibrate）的目的，是保证主臂与从臂处于相同物理位姿时，其关节位置读数也相同。这一点非常重要，因为它保证了在某一只 SO-101 上训练的神经网络策略，可以迁移到另一只 SO-101 上正常工作。

启动从臂的手动标定，运行以下命令：



In [ ]:
python lerobot/scripts/control_robot.py \
    --robot.type=so101 \
    --robot.cameras='{}' \
    --control.type=calibrate \
    --control.arms='["main_follower"]'


随后需要按照提示，依次将机械臂移动到 "中间位置、零位、旋转位置、复位位置" 四个位姿，在每个位姿按下回车确认。其中，旋转位置位于机械臂的右侧，且需要将夹爪完全张开。对主臂进行标定时，将 `--control.arms` 参数改为 `'["main_leader"]'` 即可：



In [ ]:
python lerobot/scripts/control_robot.py \
    --robot.type=so101 \
    --robot.cameras='{}' \
    --control.type=calibrate \
    --control.arms='["main_leader"]'


python lerobot/common/robot_devices/cameras/opencv.py \    --images-dir outputs/images_from_opencv_cameras


遥操作（Teleoperation）是指操作者通过某种输入设备远程控制机器人执行任务的过程。在 SO-101 平台上，遥操作采用 "主臂 - 从臂"（Leader-Follower）模式：操作者用手握住主臂并施加动作，主臂的关节角度被实时读取，LeRobot 将主臂的关节目标位置发送给从臂，使从臂实时复现主臂的运动，如图 8-4 所示。

![](assets/fig8-4_leader_arm.jpg)

图 8-4 SO-101 主臂（握把式手柄）

{: .figure-caption}

正是由于标定过程保证了主臂与从臂在相同位姿下读数一致，遥操作才能实现 "所见即所得" 的精确跟随。遥操作不仅用于实时控制机器人，更重要的用途是**采集训练数据集**—— 操作者通过遥操作演示任务，同时记录下机械臂的关节状态与摄像头图像，作为后续模仿学习训练的数据。



## 8.3.2 摄像头配置


运行后会在 `outputs/images_from_opencv_cameras` 目录下保存每个摄像头拍摄的若干帧图像，据此可以确认每个摄像头索引对应的物理摄像头（例如索引 0 对应 `camera_00`、索引 1 对应 `camera_01`）。然后，在机器人配置的 `cameras` 字段中填写实际的摄像头索引：



In [ ]:
cameras: dict[str, CameraConfig] = field(
    default_factory=lambda: {
        "wrist": OpenCVCameraConfig(
            camera_index=0,   # 腕部摄像头索引
            fps=30,
            width=640,
            height=480,
        ),
        "base": OpenCVCameraConfig(
            camera_index=1,   # 基座摄像头索引
            fps=30,
            width=640,
            height=480,
        ),
    }
)


通常建议在机械臂基座上方安装一个俯视角摄像头，观察整个任务场景；如果条件允许，还可以在腕部安装一个摄像头（SO-101 社区提供了多种腕部摄像头支架的 3D 打印模型），以便从机械臂的视角观察被抓取对象。



## 8.3.3 遥操作测试


完成摄像头配置后，运行以下命令启动遥操作。若不希望连接并显示摄像头，可以将 `--robot.cameras` 设为 `'{}'`：



In [ ]:
python lerobot/scripts/control_robot.py \
    --robot.type=so101 \
    --robot.cameras='{}' \
    --control.type=teleoperate


python lerobot/scripts/control_robot.py \    --robot.type=so101 \    --control.type=teleoperate \    --control.display_data=true


## 8.4.1 录制数据集


遥操作功能熟悉之后，就可以录制第一个训练数据集了。录制前，需要先在 Hugging Face Hub 上登录，以获取上传数据集所需的写权限令牌。登录命令如下



In [ ]:
huggingface-cli login --token ${HUGGINGFACE_TOKEN} --add-to-git-credential


然后，将 Hugging Face 用户名存入变量：



In [ ]:
HF_USER=$(huggingface-cli whoami | head -n 1)
echo $HF_USER


录制一个包含 2 个演示片段的简单数据集并上传到 Hub，运行以下命令：



In [ ]:
python lerobot/scripts/control_robot.py \
    --robot.type=so101 \
    --control.type=record \
    --control.fps=30 \
    --control.single_task="Grasp a lego block and put it in the bin." \
    --control.repo_id=${HF_USER}/so101_test \
    --control.tags='["so101","tutorial"]' \
    --control.warmup_time_s=5 \
    --control.episode_time_s=30 \
    --control.reset_time_s=30 \
    --control.num_episodes=2 \
    --control.display_data=true \
    --control.push_to_hub=true


上述命令中各参数的含义如下：

（1）`--control.fps=30`：数据采集帧率，即每秒钟记录 30 帧数据；

（2）`--control.single_task="..."`：描述当前任务的文本，用于给数据集添加语义标签；

（3）`--control.repo_id=${HF_USER}/so101_test`：数据集的仓库 ID，由用户名与仓库名组成；

（4）`--control.warmup_time_s=5`：正式开始记录前的预热时间，用于让摄像头与舵机完成同步；

（5）`--control.episode_time_s=30`：每个演示片段（Episode）的录制时长；

（6）`--control.reset_time_s=30`：每个片段结束后，复位环境所需的时间；

（7）`--control.num_episodes=2`：要录制的演示片段数量；

（8）`--control.push_to_hub=true`：录制完成后将数据集自动上传到 Hugging Face Hub。

在录制过程中，终端会不断输出类似下面的日志，反映各设备的数据处理频率：



In [ ]:
INFO 2024-08-10 15:02:58 ol_robot.py:219 dt:33.34 (30.0hz) dtRlead: 5.06 (197.5hz) dtWfoll: 0.25 (3963.7hz) dtRfoll: 6.22 (160.7hz) dtRlaptop: 32.57 (30.7hz) dtRphone: 33.84 (29.5hz)


LeRobot 将采集到的数据统一封装为 **LeRobotDataset** 格式，其核心特征可以理解为 "按时间顺序组织的、包含观测与动作的样本集合"。一个典型的数据集中，每个时间帧（frame）包含以下字段：

（1）`observation.images.cam_high` 等：摄像头图像，以 MP4 视频帧的方式存储以节省空间；

（2）`observation.state`：机械臂当前各关节的位置，用于描述状态；

（3）`action`：机械臂的目标关节位置（动作）；

（4）`episode_index` 与 `frame_index`：用于标识当前样本所属的演示片段及其在片段内的帧序号。

此外，LeRobotDataset 还通过 `stats` 保存每个特征的最大值、最小值、均值与标准差等统计信息，用于训练时的数据归一化。通过 `delta_timestamps` 机制，还可以按相对时间同时取出某个样本的前若干帧，例如 `delta_timestamps = {"observation.image": [-1, -0.5, -0.2, 0]}` 可以一次取出该样本前 1 秒、0.5 秒、0.2 秒以及当前帧共 4 帧图像，这对扩散策略等依赖历史帧的策略非常重要。

数据集在本地保存在 `~/.cache/huggingface/lerobot/{repo_id}` 目录下，例如 `data/cadene/so101_test`；上传到 Hub 后，可通过 `https://huggingface.co/datasets/${HF_USER}/so101_test` 访问。



## 8.4.3 数据集可视化与回放


录制完成后，可以在本地可视化数据集。运行以下命令会启动一个本地 Web 服务器（默认地址 `http://127.0.0.1:9090`），在浏览器中逐帧查看每个演示片段的摄像头图像与关节数据：



In [ ]:
python lerobot/scripts/visualize_dataset_html.py \
    --repo-id ${HF_USER}/so101_test \
    --local-files-only 1


LeRobot 还提供了一个非常有用的 `replay`（回放）功能，它可以在真实机器人上重放数据集中任意一个演示片段，从而检验机器人动作的重复性，并评估策略在不同机器人之间的可迁移性。回放数据集的第 0 个片段，运行以下命令



In [ ]:
python lerobot/scripts/control_robot.py \
    --robot.type=so101 \
    --control.type=replay \
    --control.fps=30 \
    --control.repo_id=${HF_USER}/so101_test \
    --control.episode=0


有了包含人类演示的数据集之后，就可以训练一个神经网络策略，让机器人学会自主完成同样的任务。这种 "从演示中学习" 的方法称为**模仿学习（Behavior Cloning）**。其基本思想是：通过最小化策略输出动作与演示动作之间的差异，学习一个从观测 $o$ 到动作 $a$ 的映射 $\pi_\theta$，其优化目标可表示为

$\theta^{*} = \arg\min_{\theta} \; \mathbb{E}_{\tau \sim \mathcal{D}} \left[ \sum_{t} \left\| a_{t} - \pi_{\theta}(o_{t}) \right\|^{2} \right] \tag{8.1}$

其中，$\tau$ 表示数据集 $\mathcal{D}$ 中的一个演示轨迹（Episode），$o_{t}$ 与 $a_{t}$ 分别是第 $t$ 帧的观测与动作，$\theta$ 为策略网络的参数。

LeRobot 内置了多种在真实世界被验证有效的策略模型，常用的是以下两种。

（1）**ACT（Action Chunking with Transformers）**：基于 Transformer 的条件变分自编码器（CVAE）架构，一次预测一段（chunk）连续的关节动作序列，具有较强的多步预测与动作平滑能力，是目前 LeRobot 上训练成本较低、稳定性较好的默认选择。

（2）**Diffusion Policy（扩散策略）**：将动作生成建模为去噪扩散过程，通过学习逐步去噪来生成动作序列，能够表达多模态的动作分布，适合抓取点多样、动作存在歧义的任务。



## 8.5.2 训练策略


使用 `lerobot/scripts/train.py` 脚本即可训练策略。下面以 ACT 策略为例，使用上节录制的数据集训练：



In [ ]:
python lerobot/scripts/train.py \
    --dataset.repo_id=${HF_USER}/so101_test \
    --policy.type=act \
    --output_dir=outputs/train/act_so101_test \
    --job_name=act_so101_test \
    --policy.device=cuda \
    --wandb.enable=true


python lerobot/scripts/train.py \    --config_path=outputs/train/act_so101_test/checkpoints/last/pretrained_model/train_config.json \    --resume=true


## 8.5.3 策略评估与自主部署


策略训练完成后，可以通过 "带策略录制" 的方式在真实机器人上评估其效果。其命令与录制训练数据非常相似，只是额外增加了一个 `--control.policy.path` 参数来指定策略检查点路径，并将数据集名称改为以 `eval` 开头以区分：



In [ ]:
python lerobot/scripts/control_robot.py \
    --robot.type=so101 \
    --control.type=record \
    --control.fps=30 \
    --control.single_task="Grasp a lego block and put it in the bin." \
    --control.repo_id=${HF_USER}/eval_act_so101_test \
    --control.tags='["tutorial"]' \
    --control.warmup_time_s=5 \
    --control.episode_time_s=30 \
    --control.reset_time_s=30 \
    --control.num_episodes=10 \
    --control.push_to_hub=true \
    --control.policy.path=outputs/train/act_so101_test/checkpoints/last/pretrained_model


除了上述命令行脚本，LeRobot 还提供了面向 Python 的编程接口，方便读者在自定义程序中直接控制 SO-101。



## 8.6.1 关节控制


下面的代码演示了如何连接 SO-101、读取当前关节位置并发送关节目标位置：



In [ ]:
from lerobot.common.robot_devices.robots.so101 import SO101Robot
import torch

# 创建机器人对象并指定从臂端口
robot = SO101Robot(port="/dev/ttyUSB0")
robot.connect()

# 读取当前 6 个关节的位置
obs = robot.capture_observation()
print("Joint positions:", obs["observation.state"])

# 发送关节位置指令（所有关节归零）
action = torch.zeros(6)
robot.send_action(action)

robot.disconnect()


其中，`SO101Robot(port=...)` 用于创建机器人对象并指定 USB 端口；`connect()` 用于建立与舵机总线的连接；`capture_observation()` 返回包含 `observation.state`（关节位置）在内的观测字典；`send_action()` 用于向机器人发送目标关节位置。



## 8.6.2 力位（重力补偿）控制


SO-101 的舵机支持反向驱动，即可以在低力矩模式下用手自由推动机械臂。利用这一特性可以实现重力补偿（反驱）模式，使机械臂在断电时仍能轻松被推动。启用该模式的代码如下：



In [ ]:
from lerobot.common.robot_devices.robots.so101 import SO101Robot
import time

robot = SO101Robot(port="/dev/ttyUSB0")
robot.connect()

# 启用反向驱动（重力补偿）模式，手臂可被手自由推动
robot.set_compliance_mode(True)

for _ in range(100):
    obs = robot.capture_observation()
    # 实时打印推动过程中的关节位置
    print(obs["observation.state"])
    time.sleep(0.02)

robot.disconnect()


本章介绍了 Hugging Face 开源机器人学习库 LeRobot 及其原生配套的低成本六自由度机械臂 SO-101。SO-101 采用串行总线舵机与 USB 通信，无须 ROS 与内核驱动即可完成环境搭建，通过 LeRobot 提供了 "标定 — 遥操作 — 数据采集 — 策略训练 — 自主部署" 的完整端到端流程。读者通过本章可以掌握低成本机械臂的安装与配置、主从臂遥操作、LeRobotDataset 数据集的采集与处理，以及基于模仿学习（ACT、扩散策略）的任务策略训练与评估方法，从而快速进入机器人学习与具身智能研究的大门。

